# Chapter 4 — Semantic Search from Scratch

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hamzafarooq/advanced-rag-from-scratch/blob/main/colab_original_notebooks/Chapter_4.ipynb)

This notebook accompanies **Chapter 4** of *Build an Advanced RAG Application (From Scratch)*.

We start with a corpus of hotel reviews and build a semantic search engine in three increasingly fast forms:

1. **Pure NumPy** — cosine similarity by hand.
2. **NumPy with normalized Euclidean distance** — same ranking, different metric.
3. **FAISS** — the production-grade vector search library.

We finish by comparing FAISS index types (Flat / HNSW / IVF-PQ) on the same query.



## 1. Setup



In [ ]:
from IPython.display import HTML, display

def set_css(*args, **kwargs):
  display(HTML('''
  <style>
    pre {
        white-space: pre-wrap;
    }
  </style>
  '''))
get_ipython().events.register('pre_run_cell', set_css)


In [ ]:
#download the required libraries

!pip install huggingface
!pip install -U datasets
#!pip install sentence-transformers
!pip install faiss-cpu
!pip install einops
!pip install sentence_transformers

In [ ]:
import pandas as pd
from datasets import load_dataset
import numpy as np
from sentence_transformers import SentenceTransformer
import torch
import scipy.spatial
from datasets import load_dataset



dataset = load_dataset("traversaal-ai-hackathon/hotel_datasets")

## 2. Load the Paris hotel reviews

The dataset is hosted on the HuggingFace Hub. The first call downloads it; subsequent calls hit the local cache.


In [ ]:
df=pd.DataFrame(dataset['train'])
df.head()

In [ ]:
df_paris = df.loc[df.locality=='Paris']

In [ ]:
df_paris.to_csv('df_paris.csv', index=False)

In [ ]:
df_paris.head()

In [ ]:
df_paris.hotel_name.value_counts()

In [ ]:
df_paris.drop_duplicates()

## 3. Embed the reviews

We use `nomic-ai/nomic-embed-text-v1.5` — a strong open-weight 768-dim embedding model. On CPU this takes a few minutes; on GPU it's much faster.


In [ ]:
model = SentenceTransformer("nomic-ai/nomic-embed-text-v1.5",trust_remote_code=True)

In [ ]:
if torch.cuda.is_available(): #A
    model = model.to('cuda')
    print("CUDA is available. The model has been moved to GPU.")
else:
    print("CUDA is not available. The model will run on CPU.")


In [ ]:
reviews = df_paris['review_text'].tolist()

In [ ]:
review_embeddings = model.encode(reviews, show_progress_bar=True)#A

In [ ]:
print(f"Embeddings shape: {review_embeddings.shape}")

In [ ]:
# Define cosine-based semantic search over the embedding matrix.

import numpy as np

def cosine_search(query_embedding, review_embeddings, k):
    # Reshape the query embedding to have a shape of (768,)
    query_embedding = query_embedding.reshape(-1)

    # Compute the dot product between the query embedding and all review embeddings
    dot_products = np.dot(review_embeddings, query_embedding)

    # Compute the L2 norms of the query embedding and review embeddings
    query_norm = np.linalg.norm(query_embedding)
    review_norms = np.linalg.norm(review_embeddings, axis=1)

    # Compute the cosine similarity scores
    cosine_similarities = dot_products / (query_norm * review_norms)

    # Get the indices of the top-k highest cosine similarity scores
    top_indices = np.argsort(-cosine_similarities)[:k]

    # Get the cosine similarity scores of the top-k similar reviews
    top_cosine_similarities = cosine_similarities[top_indices]

    return top_indices, top_cosine_similarities

In [ ]:
# Define Euclidean search on normalized vectors for comparison.

import numpy as np

def euclidean_search(query_embedding, review_embeddings, k):
    # Reshape the query embedding to have a shape of (768,)
    query_embedding = query_embedding.reshape(-1)

    # Normalize the query embedding and review embeddings
    query_embedding_normalized = query_embedding / np.linalg.norm(query_embedding)
    review_embeddings_normalized = review_embeddings / np.linalg.norm(review_embeddings, axis=1, keepdims=True)

    # Compute the Euclidean distances between the normalized query embedding and normalized review embeddings
    distances = np.linalg.norm(review_embeddings_normalized - query_embedding_normalized, axis=1)

    # Get the indices of the top-k smallest distances
    top_indices = np.argsort(distances)[:k]

    # Get the distances of the top-k similar reviews
    top_distances = distances[top_indices]

    return top_indices, top_distances


## 4. Search by hand: cosine similarity

No libraries needed. We compute dot products of L2-normalized vectors.


In [ ]:
query = "Hotel with a view of the Eiffel tower."

In [ ]:
# Embed the query and retrieve the top matches with cosine similarity.

import time
query_embedding = model.encode([query])#.astype('float32')

k = 5  # Number of similar reviews to retrieve

start_time = time.time()


indices, distances = cosine_search(query_embedding, review_embeddings, k)

biencoder_search_time = time.time() - start_time
print(f"Bi-encoder search time: {biencoder_search_time:.4f} seconds")

#indices, distances = euclidean_search(query_embedding, review_embeddings, k)


In [ ]:
print(f"Query: {query}")
print("Top hotel with similar reviews:")
for i, (idx, distance) in enumerate(zip(indices, distances), 1):
    print(f"{i}. {df_paris.iloc[idx]['hotel_name']}")
    print(f"Review: {df_paris.iloc[idx]['review_text']}")
    print(f"Distance: {distance:.4f}")
    print()


## 5. Same ranking via normalized Euclidean distance

For unit-norm vectors, `||a − b||² = 2(1 − cos(a, b))`, so Euclidean distance ranks identically to cosine similarity (just inverted).


In [ ]:
indices, distances = euclidean_search(query_embedding, review_embeddings, k)


In [ ]:
print(f"Query: {query}")
print("Top hotel with similar reviews:")
for i, (idx, distance) in enumerate(zip(indices, distances), 1):
    print(f"{i}. {df_paris.iloc[idx]['hotel_name']}")
    print(f"Review: {df_paris.iloc[idx]['review_text']}")
    print(f"Distance: {distance:.4f}")
    print()

## 6. Scale up with FAISS

NumPy is fine for a few thousand rows but quickly falls over. FAISS is purpose-built for vector search.

We build an `IndexFlatIP` (inner product) on **L2-normalized** vectors — that's mathematically equivalent to exact cosine similarity but with FAISS's optimized SIMD search.


In [ ]:
# Build a FAISS index over normalized review embeddings for fast search.

import faiss

review_embeddings = review_embeddings.astype('float32') #A



# Normalize the review embeddings
review_embeddings_normalized = review_embeddings / np.linalg.norm(review_embeddings, axis=1, keepdims=True)

# Initialize the FAISS index with cosine similarity
index = faiss.IndexFlatIP(review_embeddings_normalized.shape[1])

# Add the normalized review embeddings to the index
index.add(review_embeddings_normalized)

In [ ]:
# Normalize the query embedding
query_embedding_normalized = query_embedding / np.linalg.norm(query_embedding)

# Perform similarity search
k = 5  # Number of similar reviews to retrieve



start_time = time.time()


distances, indices = index.search(query_embedding_normalized, k)

faiss_search_time = time.time() - start_time
print(f"Faiss search time: {faiss_search_time:.4f} seconds")



In [ ]:
print(f"Query: {query}")
print("Top hotel with similar reviews using FAISS:")
for i, (idx, distance) in enumerate(zip(indices[0], distances[0]), 1):
    print(f"{i}. {df_paris.iloc[idx]['hotel_name']}")
    print(f"Review: {df_paris.iloc[idx]['review_text']}")
    print(f"Distance: {distance:.4f}")
    print()


In [ ]:
k = 25  # Number of similar reviews to retrieve

start_time = time.time()

distances, indices = index.search(query_embedding_normalized, k)

faiss_search_time = time.time() - start_time
print(f"Faiss search time: {faiss_search_time:.4f} seconds")

print(f"Query: {query}")
print("Top hotel with similar reviews using FAISS:")
for i, (idx, distance) in enumerate(zip(indices[0], distances[0]), 1):
    print(f"{i}. {df_paris.iloc[idx]['hotel_name']}")
    print(f"Review: {df_paris.iloc[idx]['review_text']}")
    print(f"Distance: {distance:.4f}")
    print()

In [ ]:
# Aggregate many retrieved reviews into hotel-level recommendations.

k = 120  # Number of similar reviews to retrieve

start_time = time.time()

distances, indices = index.search(query_embedding_normalized, k)

faiss_search_time = time.time() - start_time
print(f"Faiss search time: {faiss_search_time:.4f} seconds")

hotel_info = {}

for idx, distance in zip(indices[0], distances[0]):
    hotel_name = df_paris.iloc[idx]['hotel_name']
    if hotel_name not in hotel_info:
        hotel_info[hotel_name] = {
            'description': df_paris.iloc[idx]['hotel_description'],
            'reviews': [],
            'distances': []
        }
    hotel_info[hotel_name]['reviews'].append(df_paris.iloc[idx]['review_text'])
    hotel_info[hotel_name]['distances'].append(distance)

for hotel_name in hotel_info:
    hotel_info[hotel_name]['overall_distance'] = np.mean(hotel_info[hotel_name]['distances'])
    hotel_info[hotel_name]['num_reviews'] = len(hotel_info[hotel_name]['reviews'])

filtered_hotels = {hotel_name: hotel_data for hotel_name, hotel_data in hotel_info.items() if hotel_data['num_reviews'] >= 2}

sorted_hotels = sorted(filtered_hotels.items(), key=lambda x: x[1]['overall_distance'], reverse=True)

print(f"Query: {query}")
print("Top hotels with similar reviews using FAISS:")
for i, (hotel_name, hotel_data) in enumerate(sorted_hotels, 1):
    print(f"{i}. {hotel_name}")
    print(f"Description: {hotel_data['description']}")
    print(f"Overall Distance: {hotel_data['overall_distance']:.4f}")
    print(f"Number of Reviews: {hotel_data['num_reviews']}")
    print("Reviews:")
    for review in hotel_data['reviews']:
        print(f"- {review}")
    print()

### Aggregate by hotel

A single hotel may show up many times in the top-k. Group by hotel and rank by mean cosine to surface *places* rather than individual reviews.


In [ ]:
# Wrap the FAISS setup into reusable helper functions.

import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

def get_embeddings(data, model):
    """
    Generates embeddings for a list of text data.

    Args:
        data (list): A list of strings.
        model (SentenceTransformer): The pre-trained sentence transformer model.

    Returns:
        np.ndarray: The embeddings of the input data.
    """
    embeddings = model.encode(data, show_progress_bar=False).astype('float32')
    return embeddings

def search_faiss_index(query_embedding, faiss_index, k=5):
    """
    Performs a semantic search on a FAISS index using a query embedding.

    Args:
        query_embedding (np.ndarray): The embedding of the query string.
        faiss_index (faiss.IndexFlatIP): The FAISS index object.
        k (int): The number of nearest neighbors to retrieve.

    Returns:
        tuple: A tuple containing:
            - distances (np.ndarray): The distances of the retrieved neighbors.
            - indices (np.ndarray): The indices of the retrieved neighbors in the original data.
    """
    # Normalize the query embedding
    query_embedding_normalized = query_embedding / np.linalg.norm(query_embedding)
    distances, indices = faiss_index.search(query_embedding_normalized, k)
    return distances, indices

def create_faiss_index(embeddings):
    """
    Creates a FAISS index from a set of embeddings.

    Args:
        embeddings (np.ndarray): The embeddings to index.

    Returns:
        faiss.IndexFlatIP: The created FAISS index object.
    """
    # Normalize the embeddings
    embeddings_normalized = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)
    # Initialize the FAISS index with cosine similarity (Inner Product)
    index = faiss.IndexFlatIP(embeddings_normalized.shape[1])
    # Add the normalized embeddings to the index
    index.add(embeddings_normalized)
    return index

# Example usage (assuming 'reviews' and 'model' are defined from previous code):
reviews = df_paris['review_text'].tolist()
model = SentenceTransformer("nomic-ai/nomic-embed-text-v1.5", trust_remote_code=True)
if torch.cuda.is_available():
    model = model.to('cuda')

review_embeddings = get_embeddings(reviews, model)
faiss_index = create_faiss_index(review_embeddings)



## 7. Comparing FAISS index types

- **Flat** — exact, full-scan; slow on big corpora.
- **HNSW** — graph-based, fast & accurate, more memory.
- **IVF-PQ** — clustered + quantized, very fast, lossy.

For a small corpus the latency differences are tiny; on millions of vectors they're decisive.


In [ ]:
query = "Hotel with a view of the Eiffel tower."
query_embedding = get_embeddings([query], model)

k = 25
distances, indices = search_faiss_index(query_embedding, faiss_index, k=k)

print(f"Query: {query}")
print("Top hotel with similar reviews using FAISS:")
for i, (idx, distance) in enumerate(zip(indices[0], distances[0]), 1):
    print(f"{i}. {df_paris.iloc[idx]['hotel_name']}")
    print(f"Review: {df_paris.iloc[idx]['review_text']}")
    # Note: With normalized embeddings and IndexFlatIP, the distance is the inner product,
    # which is equal to the cosine similarity.
    print(f"Cosine Similarity: {distance:.4f}")
    print()


In [ ]:
# Build and benchmark several FAISS index types.

import time


# Flat index (L2 distance)
index_flat = faiss.IndexFlatL2(review_embeddings.shape[1])
index_flat.add(review_embeddings)

# Hierarchical Navigable Small World (HNSW) index
index_hnsw = faiss.IndexHNSWFlat(review_embeddings.shape[1], 32)
index_hnsw.add(review_embeddings)

# Inverted File (IVF) index with Product Quantization (PQ)
nlist = 100  # Number of Voronoi cells
m = 8  # Number of subquantizers
bits = 8  # Number of bits per subquantizer
index_ivfpq = faiss.IndexIVFPQ(faiss.IndexFlatL2(review_embeddings.shape[1]), review_embeddings.shape[1], nlist, m, bits)
index_ivfpq.train(review_embeddings)
index_ivfpq.add(review_embeddings)

# Perform similarity search
query = "Hotel with a view of the Eiffel Tower"
query_embedding = model.encode([query]).astype('float32')
k = 15  # Number of similar reviews to retrieve

# Flat index search
start_time = time.time()
distances_flat, indices_flat = index_flat.search(query_embedding, k)
flat_search_time = time.time() - start_time
print(f"Flat index search time: {flat_search_time:.4f} seconds")

# HNSW index search
start_time = time.time()
distances_hnsw, indices_hnsw = index_hnsw.search(query_embedding, k)
hnsw_search_time = time.time() - start_time
print(f"HNSW index search time: {hnsw_search_time:.4f} seconds")

# IVF-PQ index search
start_time = time.time()
distances_ivfpq, indices_ivfpq = index_ivfpq.search(query_embedding, k)
ivfpq_search_time = time.time() - start_time
print(f"IVF-PQ index search time: {ivfpq_search_time:.4f} seconds")



In [ ]:
# Print the top-k similar reviews for each index
print(f"\nQuery: {query}")

print("\nTop similar reviews (Flat index):")
for i, idx in enumerate(indices_flat[0], 1):
    print(f"{i}. {df_paris.iloc[idx]['hotel_name']}")
    #print(f"{i}. {df_paris.iloc[idx]['review_text']}")

print("\nTop similar reviews (HNSW index):")
for i, idx in enumerate(indices_hnsw[0], 1):
    print(f"{i}. {df_paris.iloc[idx]['hotel_name']}")
    #print(f"{i}. {df_paris.iloc[idx]['review_text']}")

print("\nTop similar reviews (IVF-PQ index):")
for i, idx in enumerate(indices_ivfpq[0], 1):
    print(f"{i}. {df_paris.iloc[idx]['hotel_name']}")
    #print(f"{i}. {df_paris.iloc[idx]['review_text']}")


In [ ]:
# Count unique hotels surfaced by each index configuration.

# Print the top-k similar reviews for each index
print(f"\nQuery: {query}")

print("\nTop similar reviews (Flat index):")
flat_hotels = set()
for i, idx in enumerate(indices_flat[0], 1):
    hotel_name = df_paris.iloc[idx]['hotel_name']
    flat_hotels.add(hotel_name)
    print(f"{i}. {hotel_name}")
print(f"Number of unique hotels (Flat index): {len(flat_hotels)}")

print("\nTop similar reviews (HNSW index):")
hnsw_hotels = set()
for i, idx in enumerate(indices_hnsw[0], 1):
    hotel_name = df_paris.iloc[idx]['hotel_name']
    hnsw_hotels.add(hotel_name)
    print(f"{i}. {hotel_name}")
print(f"Number of unique hotels (HNSW index): {len(hnsw_hotels)}")

print("\nTop similar reviews (IVF-PQ index):")
ivfpq_hotels = set()
for i, idx in enumerate(indices_ivfpq[0], 1):
    hotel_name = df_paris.iloc[idx]['hotel_name']
    ivfpq_hotels.add(hotel_name)
    print(f"{i}. {hotel_name}")
print(f"Number of unique hotels (IVF-PQ index): {len(ivfpq_hotels)}")


## What's next

Chapter 5 plugs the **decoder** (LLM) on top of these retrievals to start producing grounded answers — the first half of a full RAG pipeline. Chapter 6 wires retrieval + generation together end-to-end and adds a real vector database (Qdrant).
